# Aspect-level sentiment statistics — SemEval-2014 Laptop

Turns per-example `(aspect, sentiment)` pairs into a per-aspect summary table
(`positive`/`negative`/`neutral` counts + majority sentiment), the input the FLAN-T5
report-generation step (Tuần 4, `plans/task.txt` — Hoàng/Vinh/Hưng) will consume.

Two tables are produced, over the union of `train`/`valid`/`test`:
- **Gold**: aggregated straight from the SemEval XML labels — the ground truth.
- **Predicted**: aggregated from the fine-tuned DistilBERT model's predictions — this is the
  pipeline that will later run on data with no gold labels (e.g. the Amazon Reviews demo in
  Tuần 5, once an aspect-extraction step supplies candidate aspects there).

Comparing the two on this labeled set is a sanity check on how much aggregation noise the
model introduces before trusting it on unlabeled data.

**Kaggle setup**: add `dattm03/genai-dataset` as input (same as the fine-tuning notebooks), **and**
a second input with the fine-tuned model — upload the `distilbert-absa-model/` folder saved by
`notebooks/finetune_distilbert_semeval_laptop.ipynb` (Kaggle → Output tab of that run) as a new
Kaggle Dataset or Model, then add it here as input. GPU accelerator recommended but not required
(inference-only, dataset is small).

In [ ]:
import glob
import json
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
import xml.etree.ElementTree as ET

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


## 1. Locate the data and the fine-tuned model

Same `find_processed_dir` helper as the fine-tuning notebooks. `find_model_dir` looks for any
`/kaggle/input/.../config.json` next to model weights, then picks the one whose path contains
`absa-model` — the exact name the fine-tuning notebooks save the best-of-3-seeds model under
(`{short_name}-absa-model`). This deliberately excludes `*-absa-seed<N>*` directories: if you
created the Kaggle Model/Dataset via "New Model from notebook output", it pulls in the whole
`/kaggle/working` folder, which also contains the per-seed training checkpoints (optimizer
state and all — several hundred MB each, not needed here) alongside the actual final model.

In [ ]:
def find_processed_dir():
    search_roots = ["/kaggle/input", "../data/processed/laptop", "data/processed/laptop"]
    for root in search_roots:
        for valid_path in glob.glob(f"{root}/**/valid.xml", recursive=True) + glob.glob(f"{root}/valid.xml"):
            d = Path(valid_path).parent
            if (d / "train.xml").exists() and (d / "test.xml").exists():
                return d
    raise FileNotFoundError(
        "train.xml/valid.xml/test.xml not found. On Kaggle: add the 'dattm03/genai-dataset' dataset as "
        "notebook input. Locally: run scripts/split_dataset.py to produce data/processed/laptop/."
    )


def find_model_dir():
    search_roots = ["/kaggle/input", "../results", "results"]
    candidates = []
    for root in search_roots:
        for config_path in glob.glob(f"{root}/**/config.json", recursive=True):
            d = Path(config_path).parent
            if (d / "model.safetensors").exists() or (d / "pytorch_model.bin").exists():
                candidates.append(d)
    candidates = sorted(set(candidates), key=str)

    # Prefer the deliberately saved best-of-3-seeds model ("*-absa-model"); explicitly skip
    # per-seed training checkpoints ("*-absa-seed<N>*"), which also match "absa" but are not
    # the final model.
    final_model_candidates = [d for d in candidates if "absa-model" in str(d).lower()]
    chosen = final_model_candidates or candidates
    if not chosen:
        raise FileNotFoundError(
            "No fine-tuned model found under /kaggle/input (or results/ locally). Upload the "
            "distilbert-absa-model/ folder saved by notebooks/finetune_distilbert_semeval_laptop.ipynb "
            "as a new Kaggle Dataset/Model and add it as this notebook's input."
        )
    if len(candidates) > 1:
        print(f"Found {len(candidates)} candidate model dir(s): {candidates}")
    return chosen[0]


DATA_DIR = find_processed_dir()
MODEL_DIR = find_model_dir()
print("Using data dir:", DATA_DIR)
print("Using model dir:", MODEL_DIR)


## 2. Parse XML + flatten to (sentence, aspect, gold_sentiment)

Same logic as `src/data/semeval_loader.py` / `src/data/preprocess.py`, inlined. Uses the
**union of train/valid/test** — for a reporting table (not a held-out eval), more real data
beats a clean train/test split.

In [ ]:
@dataclass
class AspectTerm:
    term: str
    polarity: str
    start: int
    end: int


@dataclass
class Sentence:
    sentence_id: str
    text: str
    aspect_terms: list = field(default_factory=list)


def load_semeval_xml(path):
    root = ET.parse(path).getroot()
    sentences = []
    for sent_el in root.findall("sentence"):
        text = sent_el.findtext("text") or ""
        aspect_terms = []
        for term_el in sent_el.findall("./aspectTerms/aspectTerm"):
            polarity = term_el.get("polarity")
            if polarity is None:
                continue
            aspect_terms.append(
                AspectTerm(
                    term=term_el.get("term", ""),
                    polarity=polarity,
                    start=int(term_el.get("from", -1)),
                    end=int(term_el.get("to", -1)),
                )
            )
        sentences.append(
            Sentence(sentence_id=sent_el.get("id", ""), text=text, aspect_terms=aspect_terms)
        )
    return sentences


SENTIMENT_LABELS = ("positive", "negative", "neutral")

all_sentences = []
for split in ("train", "valid", "test"):
    all_sentences.extend(load_semeval_xml(DATA_DIR / f"{split}.xml"))

sentence_texts, aspect_terms, gold_sentiments = [], [], []
for sent in all_sentences:
    for term in sent.aspect_terms:
        if term.polarity not in SENTIMENT_LABELS:
            continue
        sentence_texts.append(sent.text)
        aspect_terms.append(term.term)
        gold_sentiments.append(term.polarity)

print(f"Total (sentence, aspect) examples: {len(gold_sentiments)}")


## 3. Run the fine-tuned model over every example

Batched inference, same `(sentence, aspect)` sentence-pair input the model was fine-tuned on.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

id2label = model.config.id2label
BATCH_SIZE = 32
predicted_sentiments = []

with torch.no_grad():
    for start in range(0, len(sentence_texts), BATCH_SIZE):
        batch_sentences = sentence_texts[start : start + BATCH_SIZE]
        batch_aspects = aspect_terms[start : start + BATCH_SIZE]
        inputs = tokenizer(
            batch_sentences, batch_aspects, truncation=True, padding=True, return_tensors="pt"
        ).to(device)
        logits = model(**inputs).logits
        preds = logits.argmax(dim=-1).cpu().tolist()
        predicted_sentiments.extend(id2label[p] for p in preds)

agreement = sum(p == g for p, g in zip(predicted_sentiments, gold_sentiments)) / len(gold_sentiments)
print(f"Per-example agreement with gold labels: {agreement:.4f}")


## 4. Aggregate per aspect

Same grouping logic as `src/report/aspect_stats.py` (inlined here so the notebook is
self-contained on Kaggle) — group by aspect text (case/whitespace-insensitive), count each
sentiment, take the majority. `min_mentions=2` drops aspects that only appear once (too little
signal for a report to make a claim about).

In [ ]:
@dataclass
class AspectSummary:
    aspect: str
    positive: int
    negative: int
    neutral: int
    total: int
    majority_sentiment: str


def aggregate_aspect_sentiment(records, min_mentions=1):
    counts = {}
    for aspect, sentiment in records:
        key = aspect.strip().lower()
        if not key or sentiment not in SENTIMENT_LABELS:
            continue
        counts.setdefault(key, Counter())[sentiment] += 1

    summaries = []
    for aspect, counter in counts.items():
        total = sum(counter.values())
        if total < min_mentions:
            continue
        majority_sentiment = max(
            SENTIMENT_LABELS, key=lambda label: (counter[label], -SENTIMENT_LABELS.index(label))
        )
        summaries.append(
            AspectSummary(
                aspect=aspect,
                positive=counter["positive"],
                negative=counter["negative"],
                neutral=counter["neutral"],
                total=total,
                majority_sentiment=majority_sentiment,
            )
        )

    summaries.sort(key=lambda s: (-s.total, s.aspect))
    return summaries


MIN_MENTIONS = 2

gold_summary = aggregate_aspect_sentiment(zip(aspect_terms, gold_sentiments), min_mentions=MIN_MENTIONS)
predicted_summary = aggregate_aspect_sentiment(zip(aspect_terms, predicted_sentiments), min_mentions=MIN_MENTIONS)

print(f"Distinct aspects with >= {MIN_MENTIONS} mentions: gold={len(gold_summary)}, predicted={len(predicted_summary)}")
print()
print("Top 15 aspects by mentions (predicted):")
for s in predicted_summary[:15]:
    print(f"  {s.aspect:<25} total={s.total:3d}  +{s.positive:<3d} -{s.negative:<3d} ~{s.neutral:<3d}  majority={s.majority_sentiment}")


## 5. Compare gold vs. predicted majority sentiment

Sanity check: for aspects present in both tables, how often does the model-derived majority
sentiment agree with the gold-derived majority sentiment? This is a coarser, more forgiving
metric than per-example accuracy (Section 3) — it only cares about the final report-level
claim, not every individual prediction.

In [ ]:
gold_by_aspect = {s.aspect: s for s in gold_summary}
predicted_by_aspect = {s.aspect: s for s in predicted_summary}
shared_aspects = set(gold_by_aspect) & set(predicted_by_aspect)

matches = sum(
    gold_by_aspect[a].majority_sentiment == predicted_by_aspect[a].majority_sentiment
    for a in shared_aspects
)
print(f"Aspects in both tables: {len(shared_aspects)}")
print(f"Majority-sentiment agreement: {matches}/{len(shared_aspects)} = {matches / len(shared_aspects):.4f}")


## 6. Save results

In [ ]:
out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("results")
out_dir.mkdir(parents=True, exist_ok=True)

def to_records(summary):
    return [
        {
            "aspect": s.aspect,
            "positive": s.positive,
            "negative": s.negative,
            "neutral": s.neutral,
            "total": s.total,
            "majority_sentiment": s.majority_sentiment,
        }
        for s in summary
    ]

out_path = out_dir / "aspect_stats.json"
out_path.write_text(json.dumps({
    "model_dir": str(MODEL_DIR),
    "min_mentions": MIN_MENTIONS,
    "num_examples": len(gold_sentiments),
    "per_example_agreement": agreement,
    "majority_sentiment_agreement": matches / len(shared_aspects),
    "gold": to_records(gold_summary),
    "predicted": to_records(predicted_summary),
}, indent=2))
print(f"Saved aspect stats to {out_path}")


## Next steps

- Hand off `aspect_stats.json` (the `predicted` table — the one that generalizes to unlabeled
  data) to the FLAN-T5 report-generation step (`plans/task.txt` — Hoàng/Vinh/Hưng): each row is
  a ready-made fact (`aspect`, counts, majority sentiment) to turn into report sentences, and the
  factual checker can re-derive the same counts from this file to verify the generated report.
- If `majority_sentiment_agreement` (Section 5) is noticeably lower than the per-example accuracy
  from the fine-tuning notebook, dig into which aspects flip — usually low-mention aspects near a
  count tie, which `MIN_MENTIONS` can be raised to filter out.
- Re-run with `notebooks/finetune_bert_semeval_laptop.ipynb`'s saved model once available, to see
  whether BERT changes the aggregated picture, not just the per-example metrics.
- For the Amazon Reviews demo (Tuần 5): there's no gold XML there, so only the `predicted`-table
  path applies, and it first needs an aspect-extraction step to produce candidate aspect terms
  (not yet built — see the "aspect extraction" gap noted in `plans/project-plan.md` Tuần 4).